# 面试问题：文本排序训练为什么需要 Hard Negatives，怎样构造和验证？

可以直接复述的回答是：随机负样本与 query 词面差异太大，模型很快学会简单捷径，却不会区分真正容易混淆的候选。Hard negative 通常由 BM25、旧模型或近邻召回产生，它与 query 相关但不满足关键意图。训练时必须确认它确实非相关，避免 false negative。双塔可用 pairwise/InfoNCE 目标让正样本分数高于 easy 和 hard negative。评估要在同一候选集输出逐 query 排名、MRR 或 top1，而不能只看 loss。下面手写词袋双塔、余弦分数和 hard-negative 训练。

## 真实案例：六个商品查询的正例、随机负例与难负例

难负例共享大量 query 词，但缺少关键属性，例如“无线耳机收纳盒”不是耳机，“成人运动手表”不是儿童定位手表。文本用空格表示教学分词边界。

In [1]:
import warnings  # 导入告警控制模块
import torch  # 导入 PyTorch 张量和自动微分
warnings.filterwarnings("ignore")  # 隐藏环境告警保持输出清晰
torch.set_num_threads(1)  # 固定 CPU 单线程执行
torch.manual_seed(9009)  # 固定双塔初始化
triplets = [  # 定义六个带真实意图区分的排序三元组
    ("R-01", "无线 降噪 耳机", "通勤 蓝牙 主动 降噪 耳机", "无线 降噪 耳机 收纳 盒", "咖啡 豆 深烘"),  # 同义蓝牙正例与全词配件难负例
    ("R-02", "儿童 定位 手表", "儿童 GPS 定位 通话 腕表", "儿童 定位 手表 保护 壳", "厨房 空气 炸锅"),  # 腕表正例与全词保护壳难负例
    ("R-03", "露营 便携 咖啡机", "户外 手压 便携 咖啡器", "露营 便携 咖啡机 清洁 刷", "机械 键盘 茶轴"),  # 同义场景正例与全词配件难负例
    ("R-04", "防水 运动 相机", "骑行 防水 防抖 摄像机", "防水 运动 相机 保护 壳", "婴儿 纸尿裤"),  # 摄像机正例与全词保护壳难负例
    ("R-05", "静音 无线 鼠标", "办公 静音 蓝牙 鼠标", "静音 无线 鼠标 垫 套装", "旅行 拉杆箱"),  # 蓝牙正例与全词鼠标垫难负例
    ("R-06", "磁吸 快充 充电宝", "手机 磁吸 快速 充电 电源", "磁吸 快充 充电宝 支架", "宠物 猫粮"),  # 同义电源正例与全词支架难负例
]  # 结束六个排序任务
all_texts = [text for triplet in triplets for text in triplet[1:]]  # 收集 query、正例和两类负例文本
vocabulary = sorted({token for text in all_texts for token in text.split()})  # 构建教学词表
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数索引映射
def encode(text):  # 把预分词文本编码为 token ID
    return torch.tensor([token_to_id[token] for token in text.split()], dtype=torch.long)  # 返回可变长整数序列
encoded_triplets = [(row[0], encode(row[1]), encode(row[2]), encode(row[3]), encode(row[4])) for row in triplets]  # 编码六组三元组
print("输入预览：id | query | positive | hard_negative | easy_negative")  # 输出排序数据表头
for row in triplets:  # 逐条展示六个查询
    print(" | ".join(row))  # 展示可读候选语义
print(f"词表大小={len(vocabulary)}，样本组数={len(triplets)}")  # 展示模型输入规模

输入预览：id | query | positive | hard_negative | easy_negative
R-01 | 无线 降噪 耳机 | 通勤 蓝牙 主动 降噪 耳机 | 无线 降噪 耳机 收纳 盒 | 咖啡 豆 深烘
R-02 | 儿童 定位 手表 | 儿童 GPS 定位 通话 腕表 | 儿童 定位 手表 保护 壳 | 厨房 空气 炸锅
R-03 | 露营 便携 咖啡机 | 户外 手压 便携 咖啡器 | 露营 便携 咖啡机 清洁 刷 | 机械 键盘 茶轴
R-04 | 防水 运动 相机 | 骑行 防水 防抖 摄像机 | 防水 运动 相机 保护 壳 | 婴儿 纸尿裤
R-05 | 静音 无线 鼠标 | 办公 静音 蓝牙 鼠标 | 静音 无线 鼠标 垫 套装 | 旅行 拉杆箱
R-06 | 磁吸 快充 充电宝 | 手机 磁吸 快速 充电 电源 | 磁吸 快充 充电宝 支架 | 宠物 猫粮
词表大小=58，样本组数=6


## Baseline / 基线：按 token overlap 排序

难负例刻意复用多个 query token，简单 overlap 经常把配件或错误场景排到正例之前。

In [2]:
def overlap_score(query_text, document_text):  # 计算词集合交集占查询词数比例
    query_tokens = set(query_text.split())  # 构造查询词集合
    document_tokens = set(document_text.split())  # 构造文档词集合
    return len(query_tokens & document_tokens) / len(query_tokens)  # 返回查询词覆盖率
baseline_correct = 0  # 统计 token overlap top1 命中
print("id | positive_score | hard_score | easy_score | baseline_top")  # 输出词面基线表头
for row in triplets:  # 遍历六个查询候选组
    scores = [overlap_score(row[1], row[2]), overlap_score(row[1], row[3]), overlap_score(row[1], row[4])]  # 计算三个候选词面分数
    top_index = max(range(3), key=lambda index: (scores[index], -index))  # 按分数选择第一候选并稳定处理平局
    baseline_correct += int(top_index == 0)  # 累加正例 top1
    print(f"{row[0]} | {scores[0]:.3f} | {scores[1]:.3f} | {scores[2]:.3f} | {['positive', 'hard', 'easy'][top_index]}")  # 展示词面捷径
baseline_accuracy = baseline_correct / len(triplets)  # 计算六查询 top1
print(f"token overlap top1={baseline_accuracy:.1%}")  # 汇总同候选集基线

id | positive_score | hard_score | easy_score | baseline_top
R-01 | 0.667 | 1.000 | 0.000 | hard
R-02 | 0.667 | 1.000 | 0.000 | hard
R-03 | 0.333 | 1.000 | 0.000 | hard
R-04 | 0.333 | 1.000 | 0.000 | hard
R-05 | 0.667 | 1.000 | 0.000 | hard
R-06 | 0.333 | 1.000 | 0.000 | hard
token overlap top1=0.0%


## 核心实现：共享词向量双塔与 pairwise loss

encoder 对 token embedding 求均值，余弦分数手写归一化。两种训练从完全相同初始化开始：easy-only 与 easy+hard。

In [3]:
class MeanDualEncoder(torch.nn.Module):  # 定义不依赖高层检索库的共享双塔
    def __init__(self, vocabulary_size, dimension=12):  # 初始化词向量参数
        super().__init__()  # 初始化 PyTorch 模块基类
        self.embedding = torch.nn.Parameter(torch.randn(vocabulary_size, dimension) * 0.10)  # 创建可训练 token embedding
    def encode(self, token_ids):  # 把一个可变长文本编码为向量
        return self.embedding[token_ids].mean(dim=0)  # 对文本 token embedding 做均值池化
    def forward(self, query_ids, document_ids):  # 计算 query 与 document 余弦相似度
        query_vector = self.encode(query_ids)  # 编码查询文本
        document_vector = self.encode(document_ids)  # 编码候选文档
        numerator = (query_vector * document_vector).sum()  # 计算向量点积
        denominator = torch.linalg.vector_norm(query_vector) * torch.linalg.vector_norm(document_vector) + 1e-8  # 计算余弦归一化分母
        return numerator / denominator  # 返回标量相关性分数
initial_model = MeanDualEncoder(len(vocabulary))  # 创建公平比较初始双塔
initial_state = {name: value.detach().clone() for name, value in initial_model.state_dict().items()}  # 保存共享初始词向量
def train_ranker(use_hard, steps=240, learning_rate=0.10):  # 训练 easy-only 或 hard-aware 排序器
    model = MeanDualEncoder(len(vocabulary))  # 创建独立双塔模型
    model.load_state_dict(initial_state)  # 恢复完全相同初始化
    trace = []  # 保存关键步损失、梯度与 margin
    for step in range(1, steps + 1):  # 执行固定预算全量训练
        losses = []  # 收集六查询 pairwise 损失
        margins = []  # 收集正例相对负例分数差
        for query_id, query, positive, hard, easy in encoded_triplets:  # 遍历六组三元组
            positive_score = model(query, positive)  # 真实执行 query-positive forward
            easy_score = model(query, easy)  # 真实执行 query-easy forward
            easy_margin = positive_score - easy_score  # 计算随机负例 margin
            losses.append(torch.log1p(torch.exp(-easy_margin)))  # 加入 easy pair logistic loss
            margins.append(easy_margin)  # 保存 easy margin
            if use_hard:  # 检查是否训练难负例
                hard_score = model(query, hard)  # 真实执行 query-hard forward
                hard_margin = positive_score - hard_score  # 计算难负例 margin
                losses.append(torch.log1p(torch.exp(-hard_margin)))  # 加入 hard pair logistic loss
                margins.append(hard_margin)  # 保存 hard margin
        loss = torch.stack(losses).mean()  # 计算所有 pair 平均损失
        loss.backward()  # 真实执行 backward 更新共享词向量
        gradient_norm = float(torch.linalg.vector_norm(model.embedding.grad))  # 记录词向量梯度范数
        with torch.no_grad():  # 关闭手写参数更新计算图
            model.embedding -= learning_rate * model.embedding.grad  # 使用 SGD 更新双塔 embedding
            model.embedding.grad.zero_()  # 清空本步梯度
        if step in {1, 10, 80, steps}:  # 保存关键训练节点
            trace.append((step, float(loss), float(torch.stack(margins).mean()), gradient_norm))  # 记录损失、平均 margin 和梯度
    return model, trace  # 返回训练模型与轨迹
easy_model, easy_trace = train_ranker(False)  # 训练只见随机负例的基线模型
hard_model, hard_trace = train_ranker(True)  # 训练加入难负例的主方案
print("hard-aware：step | loss | mean_margin | grad_norm")  # 输出难负例训练轨迹表头
for item in hard_trace:  # 遍历四个关键训练节点
    print(f"{item[0]:4d} | {item[1]:.5f} | {item[2]:.5f} | {item[3]:.5f}")  # 展示真实 forward/backward 过程

hard-aware：step | loss | mean_margin | grad_norm
   1 | 0.75712 | -0.06395 | 0.92338
  10 | 0.46310 | 0.62540 | 0.35786
  80 | 0.16148 | 1.75927 | 0.14298
 240 | 0.13145 | 1.96351 | 0.21034


## 六查询逐候选结果与排序指标

In [4]:
def evaluate(model):  # 在每个查询三个候选上评估 top1 和 reciprocal rank
    rows = []  # 收集逐查询分数和排名
    reciprocal_ranks = []  # 收集每个查询正例倒数排名
    for query_id, query, positive, hard, easy in encoded_triplets:  # 遍历六个评估组
        with torch.no_grad():  # 关闭排序评估梯度记录
            scores = [float(model(query, positive)), float(model(query, hard)), float(model(query, easy))]  # 计算三个候选真实双塔分数
        order = sorted(range(3), key=lambda index: (-scores[index], index))  # 按分数降序生成候选排名
        positive_rank = order.index(0) + 1  # 定位正例一基排名
        reciprocal_ranks.append(1.0 / positive_rank)  # 累加倒数排名
        rows.append((query_id, scores, order, positive_rank))  # 保存可读评估记录
    top1 = sum(row[3] == 1 for row in rows) / len(rows)  # 计算正例 top1 率
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)  # 计算六查询 MRR
    return rows, top1, mrr  # 返回逐查询记录和汇总指标
easy_rows, easy_top1, easy_mrr = evaluate(easy_model)  # 评估 easy-only 训练模型
hard_rows, hard_top1, hard_mrr = evaluate(hard_model)  # 评估 hard-aware 训练模型
print("id | easy-only[pos,hard,easy] | hard-aware[pos,hard,easy] | ranks easy->hard")  # 输出同候选集对照表头
for easy_row, hard_row in zip(easy_rows, hard_rows):  # 对齐六个查询结果
    print(f"{easy_row[0]} | {[round(value, 3) for value in easy_row[1]]} | {[round(value, 3) for value in hard_row[1]]} | {easy_row[3]}->{hard_row[3]}")  # 展示难负例如何拉开分数
print(f"easy-only top1={easy_top1:.1%} MRR={easy_mrr:.3f}")  # 汇总随机负例训练结果
print(f"hard-aware top1={hard_top1:.1%} MRR={hard_mrr:.3f}")  # 汇总难负例训练结果

id | easy-only[pos,hard,easy] | hard-aware[pos,hard,easy] | ranks easy->hard
R-01 | [0.997, 0.853, -1.0] | [0.999, -1.0, -1.0] | 1->1
R-02 | [0.999, 0.917, -1.0] | [1.0, -0.995, -0.993] | 1->1
R-03 | [1.0, 0.856, -1.0] | [1.0, -0.985, -0.974] | 1->1
R-04 | [0.999, 0.957, -0.999] | [0.999, -0.917, -0.912] | 1->1
R-05 | [0.992, 0.724, -1.0] | [0.999, -0.995, -0.983] | 1->1
R-06 | [1.0, 0.959, -1.0] | [0.995, -0.938, -0.99] | 1->1
easy-only top1=100.0% MRR=1.000
hard-aware top1=100.0% MRR=1.000


## 失败案例与修正：随机负例产生“轻松的低损失”

easy-only 模型在随机负例上 margin 很大，却没有被要求压低与 query 共享词项的配件。加入 hard pair 后，正例相对难负例 margin 才成为训练目标。

In [5]:
easy_hard_margins = [row[1][0] - row[1][1] for row in easy_rows]  # 计算 easy-only 模型正例减难负例 margin
trained_hard_margins = [row[1][0] - row[1][1] for row in hard_rows]  # 计算 hard-aware 模型正例减难负例 margin
average_easy_hard_margin = sum(easy_hard_margins) / len(easy_hard_margins)  # 汇总随机负例训练后的难例 margin
average_trained_hard_margin = sum(trained_hard_margins) / len(trained_hard_margins)  # 汇总难负例训练后的难例 margin
false_negative = ("儿童 定位 手表", "儿童 GPS 定位 通话 手表")  # 构造错误挖掘把真相关文档当负例的示意 pair
false_negative_gate = overlap_score(false_negative[0], false_negative[1]) >= 2.0 / 3.0  # 用高词面覆盖触发人工复核门禁
print("query | easy-only hard_margin | hard-aware hard_margin")  # 输出逐查询难例 margin 表头
for index, row in enumerate(triplets):  # 遍历六个查询
    print(f"{row[0]} | {easy_hard_margins[index]:.4f} | {trained_hard_margins[index]:.4f}")  # 展示主方案针对混淆候选的改善
print(f"平均 hard margin：easy-only={average_easy_hard_margin:.4f}，hard-aware={average_trained_hard_margin:.4f}")  # 汇总失败修正
print("高重合挖掘候选是否进入 false-negative 人工复核：", false_negative_gate)  # 展示难负例质量门禁

query | easy-only hard_margin | hard-aware hard_margin
R-01 | 0.1441 | 1.9994
R-02 | 0.0820 | 1.9948
R-03 | 0.1439 | 1.9850
R-04 | 0.0421 | 1.9158
R-05 | 0.2685 | 1.9938
R-06 | 0.0408 | 1.9330
平均 hard margin：easy-only=0.1202，hard-aware=1.9703
高重合挖掘候选是否进入 false-negative 人工复核： True


## 结果解读

随机负例让模型学会“咖啡豆不是耳机”这类简单边界，却没有直接监督主品与配件、儿童与成人等关键差异。逐查询分数和 hard margin 展示了难负例训练真正改变的决策边界。挖掘越难越可能混入 false negative，因此必须保留标注或规则门禁。

## 生产边界

教学双塔只有均值词袋，没有 tokenizer、batch 内负例、跨卡 all-gather 或 ANN 挖掘。生产流程需要定期用当前模型召回候选、去除点击/购买正例、抽检 false-negative rate，并在时间切分验证集评估 Recall@K、MRR 和线上成功率。hard-negative 版本必须与模型和语料快照绑定。

## 最小回归测试

In [6]:
assert len(triplets) >= 6  # 保证排序案例包含多个真实查询
assert hard_trace[-1][1] < hard_trace[0][1]  # 保证真实难负例训练降低 pairwise loss
assert hard_top1 >= easy_top1  # 保证同候选集 hard-aware top1 不低于随机负例模型
assert hard_mrr >= easy_mrr  # 保证难负例训练保持已经饱和的六查询平均倒数排名
assert average_trained_hard_margin > average_easy_hard_margin  # 保证正例相对难例的平均边界拉大
assert all(row[3] == 1 for row in hard_rows)  # 保证主方案六个正例均排在第一
assert false_negative_gate  # 保证高重合候选触发 false-negative 复核门禁